# Étape 2 : entraînement des modèles de détection de toxicité

Données : `data/clean/train_clean.csv` (colonne `comment_text` + 6 labels). Multi-label : 6 sorties sigmoïde, perte BCE avec `pos_weight`.
Trois modèles comparés : BiLSTM, DistilBERT, LAYA.

- **Entraînement** : 80 % de `train_clean.csv`.
- **Validation** : les 20 % restants (graine 42, mêmes indices pour tous les modèles).
- **Test officiel** : `test_clean.csv` avec `test_labels.csv`, **sans les lignes à `-1`** (non évaluées par Kaggle). Il n'est regardé qu'une fois, à la fin, sans aucun réglage dessus.
Les choix et le tableau comparatif final sont consignés dans `CLAUDE.md`.

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"   # doit être défini AVANT d'importer torch (repli CPU pour les opérations non gérées par MPS)

import time, json, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
from tqdm.auto import tqdm

# --- Chemins (relatifs à la racine du projet) ---
RACINE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHEMIN_CSV = RACINE / "data" / "clean" / "train_clean.csv"
CHEMIN_TEST = RACINE / "data" / "clean" / "test_clean.csv"
CHEMIN_TEST_LABELS = RACINE / "data" / "test_labels.csv"
DOSSIER_MODELES = RACINE / "models"
CHEMIN_CLAUDE_MD = RACINE / "CLAUDE.md"
LABELS = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

# --- Paramètres communs ---
SEED = 42
SEUIL = 0.5

# --- Paramètres BiLSTM ---
VOCAB_LSTM, LONGUEUR_LSTM, BATCH_LSTM = 30_000, 200, 128
EMB_LSTM, HIDDEN_LSTM, EPOCHS_LSTM, LR_LSTM = 128, 64, 3, 1e-3

# --- Paramètres LAYA ---
N_TEST_LAYA = 2000

# --- Paramètres DistilBERT ---
NOM_BERT = "distilbert-base-uncased"
N_BERT, LONGUEUR_BERT, BATCH_BERT, EPOCHS_BERT, LR_BERT = 30_000, 128, 16, 2, 2e-5

# Périphérique : CUDA (GPU NVIDIA) > MPS (Apple Silicon) > CPU
torch.set_default_dtype(torch.float32)
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps"  if torch.backends.mps.is_available() else
    "cpu"
)
print("Périphérique :", DEVICE)


def fixer_graine(seed: int = SEED):
    """Rend l'entraînement reproductible."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)


def synchroniser():
    """Attend la fin des calculs GPU pour que les chronométrages soient justes."""
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    elif DEVICE.type == "cuda":
        torch.cuda.synchronize()


fixer_graine()
DOSSIER_MODELES.mkdir(exist_ok=True)

# Données et split 80/20 (identique pour tous les modèles)

In [ ]:
df = pd.read_csv(CHEMIN_CSV)
df["comment_text"] = df["comment_text"].fillna("")
textes = df["comment_text"].to_numpy()
Y = df[LABELS].to_numpy(dtype=np.float32)

# Split fixe : les mêmes indices servent à tous les modèles
indices = np.arange(len(df))
idx_train, idx_test = train_test_split(indices, test_size=0.2, random_state=SEED)
print(f"{len(df)} lignes -> {len(idx_train)} entraînement / {len(idx_test)} test")
print("Part de positifs par label (train) :")
print(pd.Series(Y[idx_train].mean(axis=0), index=LABELS).round(4).to_string())


def calculer_pos_weight(y: np.ndarray) -> torch.Tensor:
    """pos_weight = négatifs / positifs, par label : compense le déséquilibre des classes."""
    positifs = y.sum(axis=0)
    return torch.tensor((len(y) - positifs) / np.maximum(positifs, 1), dtype=torch.float32, device=DEVICE)

# Test officiel (Kaggle) : retrait des lignes à `-1`

In [ ]:
# Test final : jamais utilisé pendant l'entraînement ni pour régler quoi que ce soit
df_off = pd.read_csv(CHEMIN_TEST)
labels_off = pd.read_csv(CHEMIN_TEST_LABELS)
assert (df_off["id"] == labels_off["id"]).all(), "test.csv et test_labels.csv ne sont pas alignés"

# Dans test_labels.csv, -1 = commentaire NON évalué par Kaggle : on le retire AVANT toute prédiction
non_evalue = (labels_off[LABELS] == -1).all(axis=1)
assert ((labels_off[LABELS] == -1).any(axis=1) == non_evalue).all(), "ligne partiellement à -1 : inattendu"
garde = ~non_evalue.to_numpy()

textes_off = df_off["comment_text"].fillna("").to_numpy()[garde]
Y_off = labels_off[LABELS].to_numpy(dtype=np.float32)[garde]
print(f"Test officiel : {len(df_off)} lignes -> {len(textes_off)} évaluées ({int(non_evalue.sum())} lignes à -1 retirées)")
print("Part de positifs par label (test officiel) :")
print(pd.Series(Y_off.mean(axis=0), index=LABELS).round(4).to_string())

# Évaluation commune

In [ ]:
RESULTATS = {}   # nom du modèle -> dict de résultats (alimenté par chaque modèle)


def evaluer(nom, y_vrai, proba, temps_train, temps_inference, jeu="validation"):
    """Précision, rappel, F1 (au seuil SEUIL) et ROC-AUC par label, + temps.

    jeu : "validation" (20 % de train) ou "test" (test officiel sans les -1).
    """
    pred = (proba >= SEUIL).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_vrai, pred, average=None, zero_division=0)
    auc = []
    for j in range(len(LABELS)):
        try:
            auc.append(roc_auc_score(y_vrai[:, j], proba[:, j]))
        except ValueError:          # label absent du jeu de test : AUC non définie
            auc.append(float("nan"))
    par_label = pd.DataFrame({"precision": p, "rappel": r, "f1": f, "roc_auc": auc}, index=LABELS)
    res = RESULTATS.setdefault(nom, {"temps_train_s": 0.0})
    if temps_train is not None:                 # le temps d'entraînement n'est donné qu'avec la validation
        res["temps_train_s"] = temps_train
    res[jeu] = {
        "par_label": par_label,
        "temps_inference_s": temps_inference,
        "ms_par_commentaire": 1000 * temps_inference / len(y_vrai),
        "n_lignes": len(y_vrai),
    }
    print(f"\n== {nom} [{jeu}] ==  inférence {temps_inference:.1f} s "
          f"({res[jeu]['ms_par_commentaire']:.2f} ms/commentaire) sur {len(y_vrai)} lignes")
    display(par_label.round(4))

# Modèle 1 : BiLSTM (jeu d'entraînement complet)

In [ ]:
# Vocabulaire construit sur le jeu d'entraînement uniquement (les textes sont déjà nettoyés : minuscules, sans ponctuation)
compteur = Counter(mot for t in textes[idx_train] for mot in t.split())
vocab = {"<pad>": 0, "<unk>": 1}
for mot, _ in compteur.most_common(VOCAB_LSTM - 2):
    vocab[mot] = len(vocab)


def encoder(liste_textes):
    """Texte -> suite d'identifiants, tronquée / complétée par des 0 à LONGUEUR_LSTM."""
    X = np.zeros((len(liste_textes), LONGUEUR_LSTM), dtype=np.int64)
    for i, t in enumerate(liste_textes):
        ids = [vocab.get(m, 1) for m in t.split()[:LONGUEUR_LSTM]] or [1]   # texte vide -> un jeton <unk>
        X[i, :len(ids)] = ids
    return torch.from_numpy(X)


X_train_l, X_test_l = encoder(textes[idx_train]), encoder(textes[idx_test])
Y_train_l = torch.from_numpy(Y[idx_train])
print("Vocabulaire :", len(vocab), "| X_train :", tuple(X_train_l.shape))

In [ ]:
class BiLSTM(nn.Module):
    """Embedding 128 -> LSTM bidirectionnel (64 par sens) -> max-pooling -> 6 logits."""

    def __init__(self, taille_vocab, emb=EMB_LSTM, hidden=HIDDEN_LSTM, n_labels=len(LABELS)):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocab, emb, padding_idx=0)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.sortie = nn.Linear(2 * hidden, n_labels)

    def forward(self, x):
        masque = (x != 0).unsqueeze(-1)                 # ignore le padding dans le pooling
        h, _ = self.lstm(self.embedding(x))
        h = h.masked_fill(~masque, -1e9)
        return self.sortie(self.dropout(h.max(dim=1).values))   # logits (la sigmoïde est dans la perte / à l'inférence)


@torch.no_grad()
def predire_lstm(modele, X, batch=512):
    modele.eval()
    sorties = [torch.sigmoid(modele(X[i:i + batch].to(DEVICE))).cpu()
               for i in tqdm(range(0, len(X), batch), desc="BiLSTM inférence")]
    return torch.cat(sorties).numpy()

In [ ]:
fixer_graine()
modele_lstm = BiLSTM(len(vocab)).to(DEVICE)
perte = nn.BCEWithLogitsLoss(pos_weight=calculer_pos_weight(Y[idx_train]))   # BCE + pos_weight
optimiseur = torch.optim.Adam(modele_lstm.parameters(), lr=LR_LSTM)
loader = DataLoader(TensorDataset(X_train_l, Y_train_l), batch_size=BATCH_LSTM, shuffle=True)

synchroniser(); t0 = time.perf_counter()
for epoque in range(EPOCHS_LSTM):
    modele_lstm.train()
    total = 0.0
    barre = tqdm(loader, desc=f"BiLSTM époque {epoque + 1}/{EPOCHS_LSTM}")
    for etape, (xb, yb) in enumerate(barre, 1):
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimiseur.zero_grad()
        l = perte(modele_lstm(xb), yb)
        l.backward()
        optimiseur.step()
        total += l.item()
        barre.set_postfix(perte=f"{total / etape:.4f}")   # perte moyenne en direct
    print(f"  perte moyenne : {total / len(loader):.4f}")
synchroniser(); temps_train_lstm = time.perf_counter() - t0

# Inférence chronométrée sur tout le jeu de test
synchroniser(); t0 = time.perf_counter()
proba_lstm = predire_lstm(modele_lstm, X_test_l)
synchroniser(); temps_inf_lstm = time.perf_counter() - t0

evaluer("BiLSTM", Y[idx_test], proba_lstm, temps_train_lstm, temps_inf_lstm)

# Sauvegarde : poids + vocabulaire (nécessaire pour réutiliser le modèle)
(DOSSIER_MODELES / "bilstm").mkdir(exist_ok=True)
torch.save(modele_lstm.state_dict(), DOSSIER_MODELES / "bilstm" / "bilstm.pt")
(DOSSIER_MODELES / "bilstm" / "vocab.json").write_text(json.dumps(vocab), encoding="utf-8")

In [ ]:
# Test officiel : BiLSTM (les modèles ne sont plus modifiés ici)
X_off_l = encoder(textes_off)
synchroniser(); t0 = time.perf_counter()
proba_lstm_off = predire_lstm(modele_lstm, X_off_l)
synchroniser(); temps_inf_lstm_off = time.perf_counter() - t0
evaluer("BiLSTM", Y_off, proba_lstm_off, None, temps_inf_lstm_off, jeu="test")

# Modèle 2 : DistilBERT (sous-échantillon stratifié de 30 000 lignes d'entraînement)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# Stratification multi-label : on stratifie sur la combinaison exacte des 6 labels ; les combinaisons rares (< 10) sont regroupées
cle = pd.Series(["".join(map(str, ligne.astype(int))) for ligne in Y[idx_train]])
rares = cle.map(cle.value_counts()) < 10
cle[rares] = "rare"

if N_BERT < len(idx_train):
    idx_bert, _ = train_test_split(idx_train, train_size=N_BERT, stratify=cle.to_numpy(), random_state=SEED)
else:
    idx_bert = idx_train
print("Lignes d'entraînement DistilBERT :", len(idx_bert))
print("Part de positifs (sous-échantillon) :", dict(zip(LABELS, Y[idx_bert].mean(axis=0).round(4))))

tokenizer = AutoTokenizer.from_pretrained(NOM_BERT)


def tokeniser(liste_textes):
    enc = tokenizer(list(liste_textes), truncation=True, padding="max_length",
                    max_length=LONGUEUR_BERT, return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]


ids_train_b, masque_train_b = tokeniser(textes[idx_bert])
ids_test_b, masque_test_b = tokeniser(textes[idx_test])
Y_train_b = torch.from_numpy(Y[idx_bert])

In [ ]:
fixer_graine()
modele_bert = AutoModelForSequenceClassification.from_pretrained(
    NOM_BERT, num_labels=len(LABELS), problem_type="multi_label_classification").to(DEVICE)   # float32 par défaut
perte_b = nn.BCEWithLogitsLoss(pos_weight=calculer_pos_weight(Y[idx_bert]))   # BCE + pos_weight (calculée sur le sous-échantillon)
optimiseur_b = torch.optim.AdamW(modele_bert.parameters(), lr=LR_BERT, weight_decay=0.01)
loader_b = DataLoader(TensorDataset(ids_train_b, masque_train_b, Y_train_b), batch_size=BATCH_BERT, shuffle=True)
n_etapes = EPOCHS_BERT * len(loader_b)
planificateur = get_linear_schedule_with_warmup(optimiseur_b, int(0.1 * n_etapes), n_etapes)   # 10 % d'échauffement

synchroniser(); t0 = time.perf_counter()
for epoque in range(EPOCHS_BERT):
    modele_bert.train()
    total = 0.0
    barre = tqdm(loader_b, desc=f"DistilBERT époque {epoque + 1}/{EPOCHS_BERT}")
    for etape, (ids, masque, yb) in enumerate(barre, 1):
        ids, masque, yb = ids.to(DEVICE), masque.to(DEVICE), yb.to(DEVICE)
        optimiseur_b.zero_grad()
        logits = modele_bert(input_ids=ids, attention_mask=masque).logits
        l = perte_b(logits, yb)
        l.backward()
        optimiseur_b.step()
        planificateur.step()
        total += l.item()
        barre.set_postfix(perte=f"{total / etape:.4f}")   # perte moyenne en direct
    print(f"  perte moyenne : {total / len(loader_b):.4f}")
synchroniser(); temps_train_bert = time.perf_counter() - t0


@torch.no_grad()
def predire_bert(modele, ids, masque, batch=64):
    modele.eval()
    sorties = [torch.sigmoid(modele(input_ids=ids[i:i + batch].to(DEVICE),
                                    attention_mask=masque[i:i + batch].to(DEVICE)).logits).cpu()
               for i in tqdm(range(0, len(ids), batch), desc="DistilBERT inférence")]
    return torch.cat(sorties).numpy()


# Inférence chronométrée sur tout le jeu de test (le même que pour les autres modèles)
synchroniser(); t0 = time.perf_counter()
proba_bert = predire_bert(modele_bert, ids_test_b, masque_test_b)
synchroniser(); temps_inf_bert = time.perf_counter() - t0

evaluer("DistilBERT", Y[idx_test], proba_bert, temps_train_bert, temps_inf_bert)

modele_bert.save_pretrained(DOSSIER_MODELES / "distilbert")
tokenizer.save_pretrained(DOSSIER_MODELES / "distilbert")

In [ ]:
# Test officiel : DistilBERT
ids_off_b, masque_off_b = tokeniser(textes_off)
synchroniser(); t0 = time.perf_counter()
proba_bert_off = predire_bert(modele_bert, ids_off_b, masque_off_b)
synchroniser(); temps_inf_bert_off = time.perf_counter() - t0
evaluer("DistilBERT", Y_off, proba_bert_off, None, temps_inf_bert_off, jeu="test")

# Modèle 3 : LAYA (Convai Innovations)

LAYA est un modèle de décision **non autorégressif** (Apache 2.0, backbone ModernBERT-large, 421 M de paramètres) : on lui donne un texte (`state`) et des questions typées, il répond à toutes en **un seul passage** avec des probabilités. Installation : `pip install laya` (dans le `.venv`).

**Adaptation au multi-label :** une question `noul` (oui/non) par label, soit 6 questions par commentaire ; la probabilité de « oui » sert de score, comme la sortie sigmoïde des autres modèles.

**Attention, version zéro-shot (sans fine-tuning).** La documentation de LAYA indique que les checkpoints de base sont proches du hasard hors de leur domaine (0,362 de précision contre 0,318 pour le hasard sur leur benchmark) et qu'ils s'utilisent comme *base à spécialiser*. De plus, leurs probabilités sont sur-confiantes et `noul` peut suivre le libellé des options plutôt que le texte. Les résultats ci-dessous sont donc un **plancher** ; le fine-tuning sur nos 6 labels n'est pas fait ici (le notebook officiel vise des GPU Kaggle 2×T4).

**Vitesse :** sur le MPS, l'inférence prend environ 0,16 à 0,35 s par commentaire (6 questions). `predict_batch` n'accélère pas, au contraire (≈ 0,3 s/commentaire avec des lots de 16, 1,3 s avec des lots de 64, mesuré). L'évaluation complète du test (≈ 32 000 lignes) prendrait ~2 h : LAYA est donc évalué sur `N_TEST_LAYA` lignes de la validation et sur `N_TEST_LAYA` lignes du test officiel. Le tableau final indique le nombre de lignes évaluées.

In [ ]:
import laya

# Une question oui/non (type "noul") par label ; l'énoncé de la question décrit le label
QUESTIONS_LAYA = {
    "toxic": "Is this comment toxic?",
    "severe_toxic": "Is this comment severely toxic?",
    "obscene": "Does this comment contain obscene language?",
    "threat": "Does this comment contain a threat?",
    "insult": "Does this comment contain an insult?",
    "identity_hate": "Does this comment contain identity-based hate?",
}
questions = {l: {"type": "noul", "instructions": q} for l, q in QUESTIONS_LAYA.items()}

agent_laya = laya.load("convaiinnovations/laya")   # checkpoint anglais (~808 Mo), téléchargé au premier appel


def predire_laya(liste_textes):
    """Probabilité de « oui » pour chaque label, texte par texte (un seul passage par texte)."""
    sortie = np.zeros((len(liste_textes), len(LABELS)), dtype=np.float32)
    for i, t in enumerate(tqdm(liste_textes, desc="LAYA")):
        reponses = agent_laya.predict(t, questions)["answers"]
        sortie[i] = [reponses[l]["noul"] for l in LABELS]
    return sortie


# Pas d'entraînement en zéro-shot : temps d'entraînement = 0.
# Inférence chronométrée sur un sous-ensemble du jeu de test (les premiers N_TEST_LAYA indices du split)
idx_laya = idx_test[:N_TEST_LAYA]
synchroniser(); t0 = time.perf_counter()
proba_laya = predire_laya(list(textes[idx_laya]))
synchroniser(); temps_inf_laya = time.perf_counter() - t0

evaluer("LAYA (zéro-shot)", Y[idx_laya], proba_laya, 0.0, temps_inf_laya, jeu="validation")

# Test officiel : mêmes N_TEST_LAYA premières lignes (déjà sans les -1)
synchroniser(); t0 = time.perf_counter()
proba_laya_off = predire_laya(list(textes_off[:N_TEST_LAYA]))
synchroniser(); temps_inf_laya_off = time.perf_counter() - t0
evaluer("LAYA (zéro-shot)", Y_off[:N_TEST_LAYA], proba_laya_off, None, temps_inf_laya_off, jeu="test")

# Tableau comparatif final

In [ ]:
# Tableau résumé : moyennes (macro) sur les 6 labels, validation et test officiel côte à côte
lignes = []
for nom, res in RESULTATS.items():
    ligne = {"modèle": nom, "entraînement (s)": res["temps_train_s"]}
    for jeu, suffixe in (("validation", "val"), ("test", "test")):
        if jeu not in res:
            continue
        moy = res[jeu]["par_label"].mean()
        ligne.update({f"précision ({suffixe})": moy["precision"], f"rappel ({suffixe})": moy["rappel"],
                      f"F1 ({suffixe})": moy["f1"], f"ROC-AUC ({suffixe})": moy["roc_auc"],
                      f"ms/commentaire ({suffixe})": res[jeu]["ms_par_commentaire"],
                      f"lignes ({suffixe})": res[jeu]["n_lignes"]})
    lignes.append(ligne)
resume = pd.DataFrame(lignes).set_index("modèle")
display(resume.round(4))

# Détail par label (validation et test)
detail = pd.concat({(nom, jeu): res[jeu]["par_label"] for nom, res in RESULTATS.items()
                    for jeu in ("validation", "test") if jeu in res}, names=["modèle", "jeu", "label"])
display(detail.round(4))

# Sauvegarde des résultats + écriture du tableau dans CLAUDE.md (entre les marqueurs)
resume.to_csv(DOSSIER_MODELES / "resultats.csv")
detail.to_csv(DOSSIER_MODELES / "resultats_par_label.csv")


def tableau_markdown(d: pd.DataFrame) -> str:
    entetes = [d.index.name] + list(d.columns)
    lignes_md = ["| " + " | ".join(entetes) + " |", "|" + "---|" * len(entetes)]
    for idx, ligne in d.iterrows():
        lignes_md.append("| " + " | ".join([str(idx)] + [f"{v:.4g}" for v in ligne]) + " |")
    return "\n".join(lignes_md)


DEBUT, FIN = "<!-- RESULTATS:DEBUT -->", "<!-- RESULTATS:FIN -->"
contenu = CHEMIN_CLAUDE_MD.read_text(encoding="utf-8")
avant, reste = contenu.split(DEBUT)
_, apres = reste.split(FIN)
nouveau = (f"{avant}{DEBUT}\n\nMacro-moyenne sur les 6 labels, seuil {SEUIL}. **val** = 20 % de train (graine {SEED}) ; "
           f"**test** = test officiel Kaggle sans les lignes à -1.\n\n"
           f"{tableau_markdown(resume)}\n\n{FIN}{apres}")
CHEMIN_CLAUDE_MD.write_text(nouveau, encoding="utf-8")
print("Tableau écrit dans", CHEMIN_CLAUDE_MD)